# LAB17 · COMERCIAL ARAGONESA S.L. · el cuadro de mando en el cuaderno
## Tu pipeline, de punta a punta, en una sola tarde

**Sesión 10 · el último cuaderno del curso · variante sin servidor**

Hoy no se aprende una herramienta nueva: **se conectan todas.** El cuadro de mando se dibuja
**aquí dentro**, sin servidor y sin puertos — y sigue siendo interactivo.

```
   tu Parquet -> DuckDB -> Plotly -> CUADRO DE MANDO
                    |                      |
                    |                      +-> la IA REDACTA -> tu AUDITAS
                    +-> la IA GENERA SQL -> DuckDB EJECUTA -> tu ANCLA VERIFICA
```

> **Las anclas de hoy:** `999.535` filas · `429.892.547,06 €` · ticket `430,09` · `10` ciudades.

---
## Paso 0 · El punto de partida: tu propio almacén · `BASE`

La primera celda crea la vista limpia **sobre el Parquet que escribiste tú** en el LAB10, no sobre
el CSV original. A partir de aquí, tu almacén ES la fuente.

In [ ]:
import duckdb, os

RUTA_PARQUET = '../datasets/salida/ventas_limpio.parquet/*.parquet'

try:
    duckdb.sql(f"CREATE OR REPLACE VIEW ventas_limpio AS "
               f"SELECT * FROM '{RUTA_PARQUET}'")
    duckdb.sql("SELECT COUNT(*) FROM ventas_limpio").fetchone()
    origen = "el Parquet maestro (tu LAB10)"
except Exception:
    # Plan B: LIMPIO-v1 sobre el CSV crudo. Mismos numeros, otro camino.
    duckdb.sql("""CREATE OR REPLACE VIEW ventas_limpio AS
    SELECT id_venta, fecha, id_cliente, id_producto, categoria,
           unidades, precio_unitario,
           CASE WHEN COALESCE(TRIM(ciudad), '') = '' THEN NULL
                ELSE UPPER(SUBSTR(TRIM(ciudad),1,1))
                     || LOWER(SUBSTR(TRIM(ciudad),2)) END AS ciudad,
           canal
    FROM '../datasets/ventas.csv'
    WHERE precio_unitario > 0""")
    origen = "el CSV crudo (plan B: revisa tu LAB10 esta noche)"

filas = duckdb.sql("SELECT COUNT(*) FROM ventas_limpio").fetchone()[0]
print(f"  Origen: {origen}")
print(f"  Filas : {filas}")
print("  ANCLA 999535 -> VALIDADO" if filas == 999535
      else "  NO CUADRA. Avisa antes de seguir.")

> 💡 Esa vista **no lee el CSV**: lee una carpeta de ficheros Parquet que escribió tu pipeline.
> **El dato ya no viene de fuera: viene de tu almacén.**

---
## Paso 1 · Los tres conjuntos del cuadro de mando · `BASE`

Un cuadro de mando necesita **tres formas**: un titular, un ranking y una evolución.

In [ ]:
import duckdb

# La vista vive en el KERNEL, no en el disco: si lo has reiniciado, se rehace
# sola sobre tu Parquet. Si tampoco está el Parquet, ejecuta el Paso 0.
try:
    duckdb.sql("SELECT 1 FROM ventas_limpio LIMIT 1")
except duckdb.CatalogException:
    duckdb.sql("CREATE OR REPLACE VIEW ventas_limpio AS SELECT * FROM "
               "'../datasets/salida/ventas_limpio.parquet/*.parquet'")

import os

os.makedirs('../datasets/salida', exist_ok=True)

duckdb.sql("""COPY (
    SELECT canal,
           ROUND(SUM(unidades*precio_unitario), 2) AS facturacion,
           COUNT(*)                                AS operaciones
    FROM ventas_limpio GROUP BY canal ORDER BY facturacion DESC
) TO '../datasets/salida/kpi_canal.csv' (HEADER)""")

duckdb.sql("""COPY (
    SELECT ciudad,
           ROUND(SUM(unidades*precio_unitario), 2) AS facturacion,
           COUNT(*)                                AS operaciones
    FROM ventas_limpio WHERE ciudad IS NOT NULL
    GROUP BY ciudad ORDER BY facturacion DESC
) TO '../datasets/salida/kpi_ciudad.csv' (HEADER)""")

duckdb.sql("""COPY (
    SELECT STRFTIME(fecha, '%Y-%m') AS mes,
           ROUND(SUM(unidades*precio_unitario), 2) AS facturacion,
           COUNT(*)                                AS operaciones
    FROM ventas_limpio GROUP BY mes ORDER BY mes
) TO '../datasets/salida/kpi_mes.csv' (HEADER)""")

for f in ['kpi_canal.csv', 'kpi_ciudad.csv', 'kpi_mes.csv']:
    n = duckdb.sql(f"SELECT COUNT(*) FROM '../datasets/salida/{f}'").fetchone()[0]
    print(f"  {f:18s} {n:3d} filas")

# Esperado: canal 3 - ciudad 10 - mes 12

⚠️ El `WHERE ciudad IS NOT NULL` **no es un adorno: es una decisión declarada.** Las 3.030 ventas
sin ciudad desaparecen de ese fichero. Dilo en tu informe.

In [ ]:
import duckdb

duckdb.sql("SELECT * FROM '../datasets/salida/kpi_canal.csv'").show()
duckdb.sql("SELECT * FROM '../datasets/salida/kpi_ciudad.csv' LIMIT 4").show()
duckdb.sql("SELECT * FROM '../datasets/salida/kpi_mes.csv'").show()

# Control: Zaragoza 146920182.71 - y apunta cual es TU mes pico

✍️ **¿Cuál es tu mes pico? ¿Coincide con lo que habrías apostado?**



---
---
# Paso 2 · El taller: AI Studio · `BASE`

Abre **`aistudio.google.com`**. No es otro chat: es **el taller con los mandos que el chat esconde**.

| Mando | Para qué | Hoy |
|---|---|---|
| Selector de modelo | con cuál trabajas | el que uses aquí es el que irá al código |
| **Temperatura** | cuánto arriesga el muestreo | **bájala a 0.2** |
| Salida estructurada | que un programa pueda consumirla | localízala |

> 🗣️ **Ni la temperatura cero garantiza determinismo.** Por eso la doctrina es **anclas y criterios**.

### Prototipa aquí el prompt del informe

```
ROL       eres analista de datos senior y escribes para direccion
CONTEXTO  (pega tus KPIs del paso 1)
TAREA     redacta 5 hallazgos, un parrafo cada uno, accionables
FORMATO   markdown, un titular en negrita por hallazgo
REGLA     usa EXCLUSIVAMENTE las cifras proporcionadas. No inventes ninguna
```

**Cuando funcione, se congela y pasa al código.** Es lo que hacen los equipos de verdad.

✍️ **¿Qué cambiaste entre el primer intento y el que te convenció?**



---
---
# Paso 3 · La IA dentro de tu pipeline · `BASE`

## 3.1 · La clave, con su liturgia

▸ **Terminal de Jupyter**: `export GEMINI_API_KEY=tu_clave` — y **reinicia el kernel**.

In [ ]:
import os

API_KEY = os.environ.get("GEMINI_API_KEY", "")
# API_KEY = "pega-aqui-tu-clave-SOLO-para-la-clase"   # <-- y BORRALA antes de entregar

print("Clave cargada." if API_KEY
      else "  FALTA la clave: export GEMINI_API_KEY=... y reinicia el kernel")

> ⚠️ La clave es una contraseña: **entorno sí, cuaderno entregado jamás.** La celda de entrega la
> busca y se niega a archivar si la encuentra.

## 3.2 · La llamada, por dentro

In [ ]:
import json, urllib.request, urllib.error

MODELO = "gemini-2.0-flash"     # <-- CONSTANTE: los nombres de modelo caducan (404)

def gemini(prompt, temperatura=0.2):
    url = (f"https://generativelanguage.googleapis.com/v1beta/models/"
           f"{MODELO}:generateContent?key={API_KEY}")
    cuerpo = {"contents": [{"parts": [{"text": prompt}]}],
              "generationConfig": {"temperature": temperatura}}
    peticion = urllib.request.Request(
        url, data=json.dumps(cuerpo).encode("utf-8"),
        headers={"Content-Type": "application/json"}, method="POST")
    try:
        with urllib.request.urlopen(peticion, timeout=60) as r:
            datos = json.load(r)          # <-- la respuesta ES JSON: tu LAB04, otra vez
        return datos["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return (f"[HTTP {e.code}]  429=cuota (espera 1 min)"
                f" · 403/400=la clave · 404=el modelo")

print(gemini("Responde solo con una palabra: capital de Aragon"))

> 💡 **La respuesta del servicio ES JSON**, y la navegas igual que tu `productos.json` del LAB04:
> `candidates` → `content` → `parts` → `text`. **Nada nuevo: lo mismo, en otro sitio.**

## 3.3 · ⭐ El patrón, verificado

In [ ]:
import duckdb

# La vista vive en el KERNEL, no en el disco: si lo has reiniciado, se rehace
# sola sobre tu Parquet. Si tampoco está el Parquet, ejecuta el Paso 0.
try:
    duckdb.sql("SELECT 1 FROM ventas_limpio LIMIT 1")
except duckdb.CatalogException:
    duckdb.sql("CREATE OR REPLACE VIEW ventas_limpio AS SELECT * FROM "
               "'../datasets/salida/ventas_limpio.parquet/*.parquet'")

def limpiar_sql(texto):
    """La IA envuelve el SQL en adornos. Higiene antes de ejecutar."""
    t = texto.strip()
    if t.startswith("```"):
        t = t.split("```")[1]
        if t.lower().startswith("sql"):
            t = t[3:]
    return t.strip().rstrip(";").strip()

ESQUEMA = duckdb.sql("DESCRIBE SELECT * FROM ventas_limpio").df().to_string(index=False)

CONTEXTO = f"""Eres un analista de datos senior.
Tengo una vista en DuckDB llamada exactamente: ventas_limpio
Su esquema es:
{ESQUEMA}
Responde SOLO con la consulta SQL, sin explicaciones y sin markdown."""

sql = limpiar_sql(gemini(
    CONTEXTO + "\nTarea: la facturacion total, es decir la suma de unidades por "
               "precio_unitario, redondeada a dos decimales."))

print("SQL GENERADO:\n", sql, "\n")

ANCLA = 429892547.06
try:
    obtenido = float(duckdb.sql(sql).fetchone()[0])
    print(f"  obtenido: {obtenido}")
    print("  ANCLA 429892547.06 -> VERIFICADO" if abs(obtenido - ANCLA) < 0.01
          else "  NO CUADRA -> a auditar el SQL. NO lo borres: leelo.")
except Exception as e:
    print(f"  El SQL no ejecuta: {type(e).__name__}. Eso TAMBIEN es un hallazgo.")

### ✍️ LA AUDITORÍA — esto es lo que se entrega

**Si salió `VERIFICADO`:** ¿qué parte del contexto se lo puso fácil?
**Si salió `NO CUADRA`:** **no lo borres.** Busca **la decisión que el modelo tomó por ti**:
¿olvidó el `ROUND`? ¿contó los `NULL`? ¿se inventó una columna?

✍️



---
---
# Paso 4 · El cuadro de mando · `BASE`
## Sin servidor, dentro del cuaderno

Plotly dibuja en el propio cuaderno y **sigue siendo interactivo**: zoom, valores al pasar por
encima y series que se encienden y apagan. Además **viaja dentro del HTML de tu entregable**.

### 4.1 · Los cuatro KPIs

In [ ]:
%pip install -q plotly

import duckdb

# La vista vive en el KERNEL, no en el disco: si lo has reiniciado, se rehace
# sola sobre tu Parquet. Si tampoco está el Parquet, ejecuta el Paso 0.
try:
    duckdb.sql("SELECT 1 FROM ventas_limpio LIMIT 1")
except duckdb.CatalogException:
    duckdb.sql("CREATE OR REPLACE VIEW ventas_limpio AS SELECT * FROM "
               "'../datasets/salida/ventas_limpio.parquet/*.parquet'")

import plotly.express as px

kpi = duckdb.sql("""SELECT COUNT(*) AS filas,
       ROUND(SUM(unidades*precio_unitario), 2)            AS facturacion,
       ROUND(SUM(unidades*precio_unitario)/COUNT(*), 2)   AS ticket,
       COUNT(DISTINCT ciudad)                             AS ciudades,
       COUNT(DISTINCT id_cliente)                         AS clientes
FROM ventas_limpio""").df().iloc[0]

def es(x, dec=2):
    """Notacion espanola: punto para miles, coma para decimales."""
    return f"{x:,.{dec}f}".replace(",", "\x00").replace(".", ",").replace("\x00", ".")

print(f"  facturacion  {es(kpi.facturacion)} EUR")
print(f"  operaciones  {es(int(kpi.filas), 0)}")
print(f"  ticket       {es(kpi.ticket)}")
print(f"  ciudades     {int(kpi.ciudades)}   clientes {es(int(kpi.clientes), 0)}")
print()
print("  ANCLAS -> 999535 - 429892547.06 - 430.09 - 10")

### 4.2 · Las tres formas: titular, ranking y evolución

In [ ]:
import duckdb

# La vista vive en el KERNEL, no en el disco: si lo has reiniciado, se rehace
# sola sobre tu Parquet. Si tampoco está el Parquet, ejecuta el Paso 0.
try:
    duckdb.sql("SELECT 1 FROM ventas_limpio LIMIT 1")
except duckdb.CatalogException:
    duckdb.sql("CREATE OR REPLACE VIEW ventas_limpio AS SELECT * FROM "
               "'../datasets/salida/ventas_limpio.parquet/*.parquet'")

import plotly.express as px

df_canal = duckdb.sql("""
    SELECT canal, ROUND(SUM(unidades*precio_unitario)/1e6,1) AS millones
    FROM ventas_limpio GROUP BY canal ORDER BY millones DESC""").df()

df_ciudad = duckdb.sql("""
    SELECT ciudad, ROUND(SUM(unidades*precio_unitario)/1e6,1) AS millones
    FROM ventas_limpio WHERE ciudad IS NOT NULL
    GROUP BY ciudad ORDER BY millones DESC""").df()

df_mes = duckdb.sql("""SELECT STRFTIME(fecha,'%Y-%m') AS mes,
           ROUND(SUM(unidades*precio_unitario)/1e6,2) AS millones,
           COUNT(*) AS operaciones
    FROM ventas_limpio GROUP BY mes ORDER BY mes""").df()

px.bar(df_canal, x="canal", y="millones", text="millones",
       title="Facturacion por canal (M EUR)").show()

px.bar(df_ciudad, x="millones", y="ciudad", orientation="h",
       title="Facturacion por ciudad (M EUR)").update_yaxes(autorange="reversed").show()

px.line(df_mes, x="mes", y="millones", markers=True,
        title="Evolucion mensual (M EUR)").show()

pico = df_mes.loc[df_mes.millones.idxmax()]
print(f"  Mes pico: {pico.mes} con {pico.millones} M EUR"
      f" y {int(pico.operaciones)} operaciones")

> 💡 **Pasa el ratón por encima de las barras y de la línea.** Los valores salen solos, y puedes
> hacer zoom arrastrando. **Eso es interactivo**, aunque no haya servidor detrás.

### 4.3 · El filtro: la misma pregunta, otro recorte

Cambia `CANALES` y vuelve a ejecutar. **Fíjate en qué número cambia y cuál no.**

In [ ]:
import duckdb

# La vista vive en el KERNEL, no en el disco: si lo has reiniciado, se rehace
# sola sobre tu Parquet. Si tampoco está el Parquet, ejecuta el Paso 0.
try:
    duckdb.sql("SELECT 1 FROM ventas_limpio LIMIT 1")
except duckdb.CatalogException:
    duckdb.sql("CREATE OR REPLACE VIEW ventas_limpio AS SELECT * FROM "
               "'../datasets/salida/ventas_limpio.parquet/*.parquet'")

CANALES = ["tienda", "web", "movil"]      # <-- quita uno y vuelve a ejecutar
SOLO_CON_CIUDAD = False        # <-- ponlo en True y mira cuantas se van

filtro = "canal IN (" + ",".join(repr(c) for c in CANALES) + ")"
if SOLO_CON_CIUDAD:
    filtro += " AND ciudad IS NOT NULL"

r = duckdb.sql(f"""SELECT COUNT(*) AS filas,
       ROUND(SUM(unidades*precio_unitario)/1e6, 1)        AS millones,
       ROUND(SUM(unidades*precio_unitario)/COUNT(*), 2)   AS ticket
FROM ventas_limpio WHERE {filtro}""").df().iloc[0]

print(f"  filtro: {filtro}")
print(f"  operaciones {int(r.filas)}  ·  {r.millones} M EUR  ·  ticket {r.ticket}")

✍️ **La facturación se desploma y el ticket medio casi no se mueve. ¿Por qué?**
*(Pista: es lo mismo que os dijeron los segmentos en el LAB07.)*


✍️ **Con `SOLO_CON_CIUDAD = True` se van 3.030 ventas. ¿Eso es limpiar o es esconder?**


### 4.4 · ⭐ El informe, y su auditoría · `BASE`

La IA redacta **a partir de KPIs que ya pasaron por tu ancla**. **No calcula nada, y ese es el
diseño:** modelos para encontrar y redactar; el motor para calcular.

In [ ]:
import json

CIFRAS = {
    "facturacion_millones": round(float(kpi.facturacion) / 1e6, 1),
    "operaciones":          int(kpi.filas),
    "ticket_medio":         float(kpi.ticket),
    "clientes_activos":     int(kpi.clientes),
    "por_canal":  df_canal.to_dict("records"),
    "por_ciudad": df_ciudad.head(4).to_dict("records"),
    "por_mes":    df_mes.to_dict("records"),
}

PROMPT = f"""ROL: eres analista de datos senior y escribes para direccion.
CONTEXTO: estas son las cifras YA VERIFICADAS de Comercial Aragonesa S.L.
{json.dumps(CIFRAS, ensure_ascii=False, indent=1)}
TAREA: redacta 5 hallazgos, un parrafo cada uno, accionables.
FORMATO: markdown, un titular en negrita por hallazgo.
REGLA: usa EXCLUSIVAMENTE las cifras proporcionadas. No inventes ninguna.
Si algo no se puede afirmar con estos datos, dilo."""

informe = gemini(PROMPT)
print(informe)

### ✍️ La auditoría del informe — criterio de éxito

**Con las cifras de arriba delante.** Tú eres el juez.

| Comprobación | ✔ |
|---|---|
| ¿Cita **solo** las cifras que le diste? | |
| ¿Alguna cifra **inventada** o redondeada raro? | |
| ¿Algún hallazgo que **los datos no sostienen**? | |
| ¿Lo firmarías y se lo darías a Dirección? | |

✍️ *(tu veredicto, con el número concreto si encontraste algo)*


> 🎯 **Si encontraste una desviación y la cuantificaste**, acabas de hacer lo que hace que a un
> perfil de datos lo contraten. La IA genera rápido; **cobra quien verifica bien.**

---
---
# 🔍 CONSULTA · Bloque final

**Cuatro preguntas, una de cada etiqueta. Se contestan hoy, en clase.**

**F1 · 🗂️ FUENTES** — *Según el material del curso, ¿por qué se eligió una **vista** y no otro fichero
para definir LIMPIO-v1? **Cítame el apartado.***

✍️


**F2 · ⚙️ MÁQUINA** *(esto no se le pregunta a nadie: se ejecuta)*

✍️ Tus números de hoy: filas · facturación · ticket · ciudades · mes pico · el veredicto del ancla.


**F3 · 🤖 ASISTENTE** — *Tengo un cuadro de mando con facturación por canal, por ciudad y por mes.
Propón **dos indicadores que NO tengo** y di qué decisión permitiría tomar cada uno.*
**Audítala:** ¿propone algo que tus datos no sostienen?

✍️


**F4 · 📝 CRITERIO** *(lo único que la IA no puede poner)*

✍️ ¿Qué pieza de esta semana llevarías mañana a tu puesto, y qué te falta para hacerlo?



---
---
# 📦 ENTREGABLE · CIERRE DEL CURSO

**`Ctrl+S` antes de nada.** Las celdas copian el fichero **guardado en disco**.

| # | Lo que tiene que estar | ¿Hecho? |
|---|---|---|
| 1 | El paso 0 en **VALIDADO** con 999.535 | |
| 2 | Los tres CSV publicados (3 · 10 · 12 filas) | |
| 3 | El patrón del paso 3.3 **y su auditoría escrita** | |
| 4 | El cuadro de mando visto, y **la auditoría del informe** | |
| 5 | El bloque de consulta contestado | |

## ① Antes de nada: ¿viaja alguna clave?

In [ ]:
import json

import glob, re

sospechosos = []
for ruta in glob.glob("*lab17*.ipynb"):
    if ".ipynb_checkpoints" in ruta:
        continue
    nb = json.load(open(ruta, encoding="utf-8"))
    for celda in nb["cells"]:
        for linea in "".join(celda["source"]).split("\n"):
            if linea.strip().startswith("#"):
                continue
            if re.search(r"AIza[0-9A-Za-z_\-]{20,}", linea):
                sospechosos.append((ruta, linea.strip()[:60]))

if sospechosos:
    print("  " + "=" * 66)
    print("   PARA. Hay algo que parece una clave dentro del cuaderno:")
    for r, l in sospechosos:
        print(f"     {r}:  {l}")
    print("   Borrala, Ctrl+S, y repite esta celda. (Y revocala en aistudio.)")
    print("  " + "=" * 66)
else:
    print("  Sin claves a la vista. Puedes archivar.")

## ② Archivar el cuaderno

In [ ]:
import glob
import json
import os

import shutil

SESION = 10

def resultados_en_disco(ruta):
    try:
        nb = json.load(open(ruta, encoding="utf-8"))
    except Exception:
        return 0, 0
    codigo = [c for c in nb["cells"] if c["cell_type"] == "code"]
    return sum(1 for c in codigo if c.get("outputs")), len(codigo)

os.makedirs("entregables", exist_ok=True)
cuadernos = [f for f in glob.glob("*lab17*.ipynb") if ".ipynb_checkpoints" not in f]

if not cuadernos:
    print("  PARA. No encuentro ningun cuaderno *lab17*.ipynb en esta carpeta.")
else:
    for c in cuadernos:
        con_salida, total = resultados_en_disco(c)
        if con_salida == 0:
            print(f"  PARA. {c} no tiene NI UN resultado guardado en disco.")
            print("        Pulsa Ctrl+S y repite esta celda.")
        else:
            destino = f"entregables/S{SESION:02d}_lab17.ipynb"
            shutil.copy(c, destino)
            print(f"  Archivado: {destino}"
                  f"   ({con_salida}/{total} celdas con resultado)")

## ③ Exportar a HTML

In [ ]:
import glob
import os

for c in glob.glob(f"entregables/S{SESION:02d}_*.ipynb"):
    salida = os.path.basename(c).replace(".ipynb", ".html")
    codigo = os.system(f'jupyter nbconvert --to html "{c}" --output "{salida}" '
                       f'--output-dir entregables 2>/dev/null')
    print(f"  {salida}  ->  {'OK' if codigo == 0 else 'FALLO'}")

## ④ El control que de verdad importa: contar los `[n]:`

In [ ]:
import glob
import os
import re

for h in sorted(glob.glob("entregables/*.html")):
    texto = open(h, encoding="utf-8", errors="ignore").read()
    n = len(re.findall(r"\[[0-9]+\]:", texto))
    print(f"  {os.path.basename(h):34s} {n:3d} celdas ejecutadas "
          f"{'' if n else '  <-- VACIO. Ctrl+S y repite.'}")

## ⑤ El paquete final del curso

In [ ]:
import glob
import os
import re

import zipfile

APELLIDO_NOMBRE = "PEREZ_Ana"        # <-- pon el tuyo ANTES de ejecutar

vacios = [os.path.basename(h) for h in glob.glob("entregables/*.html")
          if not re.findall(r"\[[0-9]+\]:",
                            open(h, encoding="utf-8", errors="ignore").read())]

if vacios:
    print("  NO EMPAQUETO: estos HTML no llevan resultados ->", ", ".join(vacios))
    print("  Ctrl+S y repite las celdas anteriores.")
else:
    nombre_zip = f"{APELLIDO_NOMBRE}_cierre.zip"
    with zipfile.ZipFile(nombre_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for raiz, carpetas, ficheros in os.walk("entregables"):
            carpetas[:] = [d for d in carpetas if d != ".ipynb_checkpoints"]
            for f in ficheros:
                z.write(os.path.join(raiz, f))
        for f in glob.glob("../datasets/salida/*.csv"):
            z.write(f, os.path.join("salida", os.path.basename(f)))
    tam = os.path.getsize(nombre_zip) / 1024
    print(f"  {nombre_zip}  ({tam:.0f} KB)  ->  subelo a Moodle")

---
---
# 🏁 La frase de salida — dila entera, en voz alta

> *«He construido un pipeline de datos real de punta a punta, con la IA como componente auditado.
> He tocado herramientas reales y las he combinado. Debo profundizar — y sé exactamente por dónde.
> **La base la tengo.»***

**Publicado en Moodle:** el **anexo del ecosistema** y el **bloque extra de continuidad** con su
proyecto. No son deberes: son la puerta siguiente.